# Day 1 / Session 3 -- Model Evaluation Exercises

Hands-on practice for Session 3. You should complete this notebook **after**
working through `D1S3_demos.ipynb`.

| # | Exercise | Difficulty | What you build |
|---|----------|------------|----------------|
| 1 | Custom Rubric | Easy (Core) | Evaluation rubric for agentic consulting AI |
| 2 | LLM Judge | Easy (Core) | Custom judge function using your rubric |
| 3 | Classification Metrics | Easy (Core) | Sklearn evaluation of LLM document classifier |
| Opt 1 | A/B Test Runner | Intermediate | Side-by-side prompt comparison with scoring |
| Opt 2 | G-Eval Implementation | Intermediate | Chain-of-thought scorer for actionability |
| Opt 3 | End-to-End Pipeline | Intermediate | Full evaluation pipeline combining all techniques |

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn python-dotenv

## Environment Setup

Run this cell to load credentials and import libraries.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import openai
import json
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

## Session Recap

In the demos you built a full evaluation toolkit: scoring rubrics, an LLM-as-a-Judge
function, benchmarking across model configs, cost/latency analysis, an A/B testing
framework, scikit-learn classification metrics, and G-Eval chain-of-thought scoring.
Now you will apply that knowledge to build your own evaluation tools from scratch.

---
## Exercise 1: Create a Custom Agentic Evaluation Rubric (Easy -- Core)

Build a specialized evaluation rubric for an **agentic consulting AI** system.
Three criteria are provided for you. You need to add two more.

> ** Key Concept:** A rubric defines *what* you are evaluating and *how* to
> score it (1-5 scale). This is the foundation for all automated evaluation.

**What to do:**
1. Read the 3 provided criteria to understand the pattern
2. Add 2 more criteria of your own (suggestions below)
3. Run the cell to see the full rubric printed

**Suggested criteria to add:**
- `client_readiness` -- Is the output polished enough for a client-facing deliverable?
- `recommendation_quality` -- Are the recommendations specific, prioritized, and evidence-backed?

**Reference:** Demo 1 for the `create_scoring_rubric()` function structure.

In [ ]:
# Exercise 1: Create a Custom Agentic Evaluation Rubric
#
# 3 criteria are provided. Add 2 more following the same pattern.

def create_agentic_rubric():
    """Build and return a rubric tailored to agentic consulting AI.

    Returns:
        dict: {criterion_name: {"description": str, "1": str, ..., "5": str}}
    """
    rubric = {}

    # [OK] Criterion 1: tool_usage_accuracy (provided)
    rubric["tool_usage_accuracy"] = {
        "description": "Did the AI select the right analytical tools/frameworks? (1-5)",
        "1": "No relevant tool or framework referenced",
        "2": "Tool mentioned but applied incorrectly or superficially",
        "3": "Appropriate tool selected; application is adequate but not thorough",
        "4": "Strong tool selection with solid application and minor gaps",
        "5": "Optimal tool/framework chosen and applied with expert-level depth"
    }

    # [OK] Criterion 2: reasoning_quality (provided)
    rubric["reasoning_quality"] = {
        "description": "How rigorous and logical is the analytical reasoning? (1-5)",
        "1": "Reasoning is absent or logically flawed",
        "2": "Some reasoning present but with significant gaps or unsupported claims",
        "3": "Reasoning is sound but lacks depth or nuance",
        "4": "Rigorous reasoning with clear logic chain and supporting evidence",
        "5": "Exceptional analytical rigor; could withstand partner-level challenge"
    }

    # [OK] Criterion 3: mece_decomposition (provided)
    rubric["mece_decomposition"] = {
        "description": "Does the response decompose the problem in a MECE way? (1-5)",
        "1": "No structure; stream-of-consciousness answer",
        "2": "Some structure but overlapping or missing categories",
        "3": "Reasonable structure; mostly MECE with minor overlaps",
        "4": "Clear MECE breakdown with well-defined, non-overlapping categories",
        "5": "Textbook MECE decomposition; exhaustive and mutually exclusive"
    }

    # TODO 1: Add a 4th criterion (e.g., "client_readiness")
    #  Hint: Follow the same pattern -- description + levels 1 through 5
    # rubric["client_readiness"] = {
    #     "description": "Is the output polished enough for a client deliverable? (1-5)",
    #     "1": "Not suitable for any external audience",
    #     "2": "...",
    #     "3": "...",
    #     "4": "...",
    #     "5": "Ready to present to the C-suite as-is"
    # }


    # TODO 2: Add a 5th criterion (e.g., "recommendation_quality")
    #  Hint: Think about what makes recommendations actionable


    return rubric


# Display the rubric
rubric = create_agentic_rubric()
for criterion, details in rubric.items():
    print(f"\n{criterion.upper()}: {details['description']}")
    for level in ["1", "2", "3", "4", "5"]:
        print(f"  {level}: {details[level]}")

### Hints

<details>
<summary><strong>Hint 1:</strong> client_readiness example</summary>

```python
rubric["client_readiness"] = {
    "description": "Is the output polished enough for a client deliverable? (1-5)",
    "1": "Not suitable for any external audience",
    "2": "Needs major rework before sharing with the client team",
    "3": "Acceptable draft; needs editing before client presentation",
    "4": "Near-final quality; minor polish needed",
    "5": "Ready to present to the C-suite as-is"
}
```
</details>

<details>
<summary><strong>Hint 2:</strong> recommendation_quality example</summary>

```python
rubric["recommendation_quality"] = {
    "description": "Are recommendations specific, prioritized, and evidence-backed? (1-5)",
    "1": "No actionable recommendations provided",
    "2": "Generic recommendations without prioritization or evidence",
    "3": "Reasonable recommendations but lacking specificity or data backing",
    "4": "Specific, prioritized recommendations with supporting evidence",
    "5": "Board-ready recommendations with clear ROI estimates and implementation roadmap"
}
```
</details>

### Expected Output

When your implementation is complete, running the cell above should produce
output similar to the following (your descriptions will differ):

```
TOOL_USAGE_ACCURACY: Did the AI select the right analytical tools/frameworks? (1-5)
 1: No relevant tool or framework referenced
 2: Tool mentioned but applied incorrectly or superficially
 3: Appropriate tool selected; application is adequate but not thorough
 4: Strong tool selection with solid application and minor gaps
 5: Optimal tool/framework chosen and applied with expert-level depth

REASONING_QUALITY: How rigorous and logical is the analytical reasoning? (1-5)
 1: Reasoning is absent or logically flawed
 2: Some reasoning present but with significant gaps or unsupported claims
 3: Reasoning is sound but lacks depth or nuance
 4: Rigorous reasoning with clear logic chain and supporting evidence
 5: Exceptional analytical rigor; could withstand partner-level challenge

MECE_DECOMPOSITION: Does the response decompose the problem in a MECE way? (1-5)
 1: No structure; stream-of-consciousness answer
 2: Some structure but overlapping or missing categories
 3: Reasonable structure; mostly MECE with minor overlaps
 4: Clear MECE breakdown with well-defined, non-overlapping categories
 5: Textbook MECE decomposition; exhaustive and mutually exclusive

CLIENT_READINESS: Is the output polished enough for a client deliverable? (1-5)
 1: Not suitable for any external audience
 2: Needs major rework before sharing with the client team
 3: Acceptable draft; needs editing before client presentation
 4: Near-final quality; minor polish needed
 5: Ready to present to the C-suite as-is

RECOMMENDATION_QUALITY: Are recommendations specific, prioritized, and evidence-backed? (1-5)
 1: No actionable recommendations provided
 2: Generic recommendations without prioritization or evidence
 3: Reasonable recommendations but lacking specificity or data backing
 4: Specific, prioritized recommendations with supporting evidence
 5: Board-ready recommendations with clear ROI estimates and implementation roadmap
```

---
## Exercise 2: Build Your Own LLM Judge (Easy -- Core)

Now that you have a rubric, wire it into an automated LLM judge. Most of
the code is provided -- you just need to write the two test responses
and uncomment the test code.

> ** Key Concept:** LLM-as-a-Judge = send a response + rubric criteria to the
> LLM and ask it to score each criterion. Using `response_format={"type": "json_object"}`
> forces the LLM to return valid JSON.

**What to do:**
1. Review the provided `custom_judge` function to understand the pattern
2. Write a high-quality and a mediocre test response
3. Run the judge on both and compare scores

**Reference:** Demo 2 for the LLM judge pattern.

In [ ]:
# Exercise 2: Build Your Own LLM Judge
#
# The judge function is provided. You just need to write the test responses below.

def custom_judge(question, response_text, rubric):
    """Score a response using your custom rubric via LLM-as-a-Judge.

    Args:
        question: The original client/consulting question
        response_text: The AI-generated response to evaluate
        rubric: Dict from create_agentic_rubric()

    Returns:
        dict: {criterion_name: {"score": int, "reasoning": str}}
    """
    # Build criteria list from rubric descriptions
    criteria_list = "\n".join(f"- {name}: {details['description']}" for name, details in rubric.items())

    # Build the judge prompt
    judge_prompt = f"""You are a McKinsey engagement manager evaluating AI-generated consulting analysis.

Question asked: {question}

Response to evaluate:
{response_text}

Rate the response on each criterion below (1-5 scale):
{criteria_list}

Return ONLY a JSON object. Each key is a criterion name, each value is an object
with "score" (integer 1-5) and "reasoning" (one short sentence)."""

    # Call the API with JSON response format
    client = openai.OpenAI()
    evaluation = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": judge_prompt}],
        response_format={"type": "json_object"},
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "600"))
    )

    return json.loads(evaluation.choices[0].message.content)

In [ ]:
# Exercise 2 (continued): Test your judge on two responses
#
# TODO: Write a good response and a mediocre response, then uncomment the test code.

consulting_question = "How should a healthcare company reduce operational costs by 15% over 2 years?"

# TODO 1: Write a HIGH-QUALITY response (2-3 sentences with specific numbers, frameworks, structure)
#  Hint: Include things like "Using zero-based budgeting, target 8-12% savings in
#          supply chain by renegotiating vendor contracts. Phase 1 (Q1-Q2): audit..."
good_response = """
YOUR HIGH-QUALITY RESPONSE HERE
"""

# TODO 2: Write a MEDIOCRE response (2-3 vague sentences with no specifics)
#  Hint: Use phrases like "improve efficiency" and "consider various options"
mediocre_response = """
YOUR MEDIOCRE RESPONSE HERE
"""

# Uncomment to run the judge on both responses:
# rubric = create_agentic_rubric()
#
# print("=" * 60)
# print("GOOD RESPONSE SCORES:")
# print("=" * 60)
# good_scores = custom_judge(consulting_question, good_response, rubric)
# print(json.dumps(good_scores, indent=2))
#
# print("\n" + "=" * 60)
# print("MEDIOCRE RESPONSE SCORES:")
# print("=" * 60)
# mediocre_scores = custom_judge(consulting_question, mediocre_response, rubric)
# print(json.dumps(mediocre_scores, indent=2))

### Hints

<details>
<summary><strong>Hint 1:</strong> Example good response</summary>

```python
good_response = """Using a MECE framework, I recommend three workstreams:
1. Supply chain optimization (target 8-12% savings): Renegotiate top 20 vendor contracts,
   implement GPO purchasing. Timeline: Q1-Q2.
2. Workforce productivity (target 3-5% savings): Deploy scheduling software to reduce
   overtime by 25%. Timeline: Q2-Q3.
3. Administrative consolidation (target 2-3% savings): Centralize billing and coding
   operations across facilities. Timeline: Q3-Q4.
Total projected savings: 13-20%, exceeding the 15% target."""
```
</details>

<details>
<summary><strong>Hint 2:</strong> Example mediocre response</summary>

```python
mediocre_response = """The healthcare company should look at ways to improve efficiency
and reduce costs. They could consider various options like cutting expenses and
streamlining operations. It would be good to explore different areas for savings."""
```
</details>

---
## Exercise 3: Classification Evaluation with Sklearn (Easy -- Core)

Use an LLM to classify consulting documents by type, then evaluate the accuracy
using scikit-learn metrics.

The test dataset and categories are provided. You just need to:
1. Uncomment and run the classification loop
2. Uncomment and run the metrics/confusion matrix code

> ** Key Concept:** Even though LLMs are great at classification, you still need
> to measure accuracy with standard ML metrics (precision, recall, F1) to know
> *how good* the classifier actually is.

**Reference:** Demo 6 for the sklearn metrics pattern.

In [ ]:
# Exercise 3a: Labeled dataset of consulting document types (provided)

categories = ["MARKET_ANALYSIS", "DUE_DILIGENCE", "ORG_DESIGN", "COST_REDUCTION", "DIGITAL_STRATEGY"]

test_data = [
    # MARKET_ANALYSIS samples
    {"text": "The addressable market for electric vehicles in Southeast Asia is projected to reach $45B by 2030, driven by government subsidies and rising middle-class demand.", "true_label": "MARKET_ANALYSIS"},
    {"text": "Analysis of the US pet food market reveals a shift toward premium and organic segments, with the top 5 players controlling 62% market share.", "true_label": "MARKET_ANALYSIS"},

    # DUE_DILIGENCE samples
    {"text": "The target company reported $120M in revenue with 18% EBITDA margins. Key risk: customer concentration with top 3 clients representing 55% of revenue.", "true_label": "DUE_DILIGENCE"},
    {"text": "Synergy estimates suggest $30M in annual savings from procurement consolidation and $15M from headcount optimization post-merger.", "true_label": "DUE_DILIGENCE"},

    # ORG_DESIGN samples
    {"text": "Recommend reducing CEO span of control from 14 to 8 direct reports by creating two new SVP roles for Commercial and Operations.", "true_label": "ORG_DESIGN"},
    {"text": "Transitioning to a shared services model for HR and Finance could reduce administrative FTEs by 25% across the three business units.", "true_label": "ORG_DESIGN"},

    # COST_REDUCTION samples
    {"text": "Zero-based budgeting analysis identified $40M in addressable spend, with quick wins in travel, contractors, and software licenses.", "true_label": "COST_REDUCTION"},
    {"text": "Procurement optimization through volume consolidation and supplier rationalization can reduce COGS by 6-8% within 12 months.", "true_label": "COST_REDUCTION"},

    # DIGITAL_STRATEGY samples
    {"text": "Deploying an AI-powered demand forecasting system could reduce inventory carrying costs by 20% while improving product availability.", "true_label": "DIGITAL_STRATEGY"},
    {"text": "The client should build a composable commerce platform using headless architecture to enable rapid experimentation across digital channels.", "true_label": "DIGITAL_STRATEGY"},
]

print(f"Dataset: {len(test_data)} samples across {len(categories)} categories")

In [ ]:
# Exercise 3b: Classify each sample with the LLM
#
# TODO: Uncomment the code below and run it. The classification loop is provided.
#  Hint: This uses the same API call pattern as the few-shot classifier in Session 2,
#          but with a system prompt instead of few-shot examples.

client = openai.OpenAI()

true_labels = []
predicted_labels = []

for item in test_data:
    # Classify using the LLM (uncomment all lines below)
    # response = client.chat.completions.create(
    #     model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    #     messages=[
    #         {"role": "system", "content": "You are a consulting document classifier. Classify the document type. Respond with EXACTLY one of: MARKET_ANALYSIS, DUE_DILIGENCE, ORG_DESIGN, COST_REDUCTION, DIGITAL_STRATEGY"},
    #         {"role": "user", "content": item["text"]}
    #     ],
    #     max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "10")),
    #     temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
    # )
    # prediction = response.choices[0].message.content.strip().upper()
    # true_labels.append(item["true_label"])
    # predicted_labels.append(prediction)
    # print(f"  {prediction:20s} (true: {item['true_label']})")
    pass  # <-- remove this line when you uncomment the code above

print(f"\nClassified {len(predicted_labels)} samples")

In [ ]:
# Exercise 3c: Compute metrics and display confusion matrix
#
# TODO: Uncomment the code below and run it. Everything is provided.

# Classification report
# print("CLASSIFICATION REPORT -- Consulting Document Types")
# print("=" * 60)
# print(classification_report(true_labels, predicted_labels, labels=categories))

# Confusion matrix visualization
# cm = confusion_matrix(true_labels, predicted_labels, labels=categories)
# fig, ax = plt.subplots(figsize=(8, 6))
# im = ax.imshow(cm, cmap="Blues")
# ax.set_xticks(range(len(categories)))
# ax.set_yticks(range(len(categories)))
# ax.set_xticklabels(categories, rotation=45, ha="right")
# ax.set_yticklabels(categories)
# ax.set_xlabel("Predicted Label")
# ax.set_ylabel("True Label")
# ax.set_title("Confusion Matrix -- Consulting Document Classification")
# for i in range(len(categories)):
#     for j in range(len(categories)):
#         ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=12,
#                 color="white" if cm[i, j] > cm.max() / 2 else "black")
# plt.colorbar(im)
# plt.tight_layout()
# plt.show()

### Hints

<details>
<summary><strong>Hint 1:</strong> How to complete Exercise 3b</summary>

Just uncomment all the lines inside the `for` loop and remove the `pass` statement.
The code is already complete -- you just need to activate it.
</details>

<details>
<summary><strong>Hint 2:</strong> How to complete Exercise 3c</summary>

Same approach: uncomment all the lines. The `classification_report` and
`confusion_matrix` functions from sklearn are already imported.
</details>

<details>
<summary><strong>Hint 3:</strong> What if some predictions don't match categories?</summary>

If the LLM returns something like "MARKET ANALYSIS" (with a space) instead of
"MARKET_ANALYSIS", add `.replace(" ", "_")` after `.strip().upper()`:
```python
prediction = response.choices[0].message.content.strip().upper().replace(" ", "_")
```
</details>

---
# Optional Exercises (Intermediate)

These exercises are for students who finish the core exercises early or want
additional practice. They are at an intermediate level.

---
## Optional Exercise 1 (Intermediate): A/B Test Runner

Build a simplified A/B testing function that compares two system prompts
on the same set of test questions, scores both with your custom judge,
and declares a winner.

> ** Hint:** The flow is: for each question -> generate response A -> generate
> response B -> score both with `custom_judge` -> compare average scores.

**Reference:** Demo 5 for the A/B testing pattern.

In [ ]:
# Optional Exercise 1: A/B Test Runner
#
# TODO Step 1: Define the function signature
# TODO Step 2: Loop through test questions
# TODO Step 3: Generate responses from both prompts
# TODO Step 4: Score both with custom_judge
# TODO Step 5: Aggregate scores and determine winner

def run_ab_test(prompt_a, prompt_b, test_questions, rubric):
 """Run an A/B test comparing two system prompts.

 Args:
 prompt_a: System prompt for configuration A
 prompt_b: System prompt for configuration B
 test_questions: List of consulting questions to test
 rubric: Your evaluation rubric from Exercise 1

 Returns:
 dict: {"config_a_avg": float, "config_b_avg": float, "winner": str, "details": list}
 """
 client = openai.OpenAI()
 details = []

 # TODO: Loop through test_questions
 # For each question:
 # 1. Generate response with prompt_a as system message
 # 2. Generate response with prompt_b as system message
 # 3. Score both responses with custom_judge(question, response, rubric)
 # 4. Append scores to details list

 # TODO: Compute average scores across all criteria for each config
 # TODO: Determine winner
 # TODO: Return summary dict

 pass # Remove this line when you implement


# Test with two system prompts
prompt_a = "You are a McKinsey senior consultant. Provide structured, MECE analysis with specific data points and a phased implementation roadmap."
prompt_b = "You are a helpful assistant. Answer the question."

test_questions = [
 "How should a bank reduce customer churn by 20%?",
 "What is the optimal pricing strategy for a SaaS product entering the enterprise market?",
]

# rubric = create_agentic_rubric() # Use your rubric from Exercise 1
# results = run_ab_test(prompt_a, prompt_b, test_questions, rubric)
# print(json.dumps(results, indent=2))

### Hints

- Use `os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")` for the model name.
- To compute average scores, extract all `"score"` values from the judge result dict.
- The winner is the config with the higher average score across all criteria and questions.
- Keep `max_tokens` reasonable (300) to control cost -- this function makes many API calls.

---
## Optional Exercise 2 (Intermediate): G-Eval Implementation

Implement your own G-Eval scorer for a custom criterion: **consulting_actionability** --
does the response provide specific, implementable next steps?

> ** Hint:** G-Eval = ask the LLM to reason step-by-step about a criterion
> *before* assigning a score. The chain-of-thought reasoning improves scoring
> accuracy compared to asking for a score directly.

**Reference:** Demo 7 for the G-Eval pattern.

In [ ]:
# Optional Exercise 2: G-Eval for Consulting Actionability
#
# TODO Step 1: Define the g_eval_actionability function
# TODO Step 2: Build a chain-of-thought prompt for actionability scoring
# TODO Step 3: Call the API with JSON response format and temperature=0
# TODO Step 4: Parse result and add normalized_score
# TODO Step 5: Test on 3 responses of varying quality

def g_eval_actionability(question, response_text):
 """Score response actionability using G-Eval chain-of-thought.

 Criterion: Does the response provide specific, implementable next steps
 with clear owners, timelines, and success metrics?

 Returns:
 dict: {"reasoning": str, "score": int, "normalized_score": float}
 """
 # TODO: Build the prompt following the G-Eval pattern from Demo 7
 # Key elements:
 # - Define the criterion clearly
 # - Ask for step-by-step reasoning BEFORE scoring
 # - Request JSON output with "reasoning" and "score" keys

 # prompt = f"""You are an expert evaluator assessing consulting analysis.
 # Evaluation Criterion: Consulting Actionability
 # Description: ...
 # ...
 # Respond in JSON with keys: "reasoning" (2-3 sentences) and "score" (integer 1-5)."""

 # client = openai.OpenAI()
 # response = client.chat.completions.create(
 # model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 # messages=[{"role": "user", "content": prompt}],
 # response_format={"type": "json_object"},
 # temperature=0,
 # max_tokens=300
 # )
 # result = json.loads(response.choices[0].message.content)
 # result["normalized_score"] = result["score"] / 5.0
 # return result

 pass # Remove this line when you implement


# Test on 3 responses of varying quality
test_question = "How should a logistics company improve last-mile delivery efficiency?"

high_actionability = """
# TODO: Write a response with specific steps, owners, timelines, and metrics
# Example: "1. Deploy route optimization software (LogiNext) by Q2 -- Operations VP owns this.
# Expected 15% reduction in delivery time. Budget: $200K."
"""

medium_actionability = """
# TODO: Write a response with some steps but vague on details
# Example: "Consider improving routing and investing in technology. The operations team
# should look into various solutions over the coming months."
"""

low_actionability = """
# TODO: Write a response that is abstract with no concrete steps
# Example: "Last-mile delivery is a complex challenge that requires holistic thinking
# and cross-functional alignment to address effectively."
"""

# for label, response in [("HIGH", high_actionability), ("MEDIUM", medium_actionability), ("LOW", low_actionability)]:
# result = g_eval_actionability(test_question, response)
# print(f"\n{label} ACTIONABILITY:")
# print(f" Score: {result['score']}/5 ({result['normalized_score']:.2f})")
# print(f" Reasoning: {result['reasoning']}")

### Hints

- The criterion description should mention: specific next steps, named owners/roles, timelines, and measurable success metrics.
- The chain-of-thought prompt should say "First, think step-by-step about how well the output provides implementable actions. Then assign a score."
- Scores should clearly separate the three quality levels: expect ~5 for high, ~3 for medium, ~1-2 for low.
- If scores are too close together, make your test responses more extreme.

---
## Optional Exercise 3 (Intermediate): End-to-End Evaluation Pipeline

Build a complete evaluation pipeline that generates responses, scores them
with both LLM judge AND G-Eval, and produces a summary report.

> ** Hint:** This combines everything: generate response -> score with
> `custom_judge` -> score with G-Eval -> aggregate into a DataFrame.
> Each test question produces one row in the final report.

**This exercise combines concepts from all 7 demos and the core exercises.**

In [ ]:
# Optional Exercise 3: End-to-End Evaluation Pipeline
#
# This combines: Exercise 1 (rubric) + Exercise 2 (judge) + Demo 7 (G-Eval)

def evaluate_ai_system(system_prompt, test_cases, rubric):
 """End-to-end evaluation: generate responses, score with judge + G-Eval, aggregate.

 Args:
 system_prompt: System prompt for the AI being evaluated
 test_cases: List of consulting questions to test
 rubric: Your evaluation rubric from Exercise 1

 Returns:
 pd.DataFrame: Summary with columns for each criterion + G-Eval scores
 """
 client = openai.OpenAI()
 all_results = []

 for question in test_cases:
 # TODO: Generate response using system_prompt
 # response = client.chat.completions.create(
 # model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 # messages=[
 # {"role": "system", "content": system_prompt},
 # {"role": "user", "content": question}
 # ],
 # max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
 # )
 # response_text = response.choices[0].message.content

 # TODO: Score with custom_judge
 # judge_scores = custom_judge(question, response_text, rubric)

 # TODO: Score with G-Eval (use g_eval from Demo 7 or g_eval_actionability)
 # For G-Eval, you can evaluate on "Coherence" and "Relevance" criteria

 # TODO: Combine scores into a single row dict
 # row = {"question": question[:50] + "..."}
 # for criterion, data in judge_scores.items():
 # row[f"judge_{criterion}"] = data["score"]
 # row["geval_coherence"] = ...
 # row["geval_relevance"] = ...
 # all_results.append(row)
 pass # Remove this line when you implement

 # TODO: Build DataFrame and compute summary statistics
 # df = pd.DataFrame(all_results)
 # print("\nPer-question scores:")
 # print(df.to_string(index=False))
 #
 # # Aggregate summary
 # score_cols = [c for c in df.columns if c != "question"]
 # summary = df[score_cols].agg(["mean", "min", "max"]).round(2)
 # print("\nAggregate Summary:")
 # print(summary)
 # return df

 pass # Remove this line when you implement


# Test the pipeline
system_prompt = "You are a McKinsey senior consultant. Provide structured, MECE, data-driven strategic analysis with specific recommendations."

test_cases = [
 "How should a regional bank compete with fintech disruptors?",
 "What is the optimal supply chain strategy for a D2C brand scaling internationally?",
 "How can a hospital system reduce readmission rates by 25%?",
]

# rubric = create_agentic_rubric()
# results_df = evaluate_ai_system(system_prompt, test_cases, rubric)

### Hints

- This function makes many API calls (1 generation + 1 judge + 2 G-Eval per test case). Keep test_cases short.
- Use `os.getenv()` for model name and max_tokens throughout.
- The DataFrame should have one row per test case and one column per score.
- For the G-Eval columns, you can reuse the `g_eval()` function from Demo 7 by copying it here, or import the criteria dict and call it directly.
- Aggregate with `df[score_cols].agg(["mean", "min", "max"])` for a clean summary.

---
## Wrap-up

You have built:

1. **Exercise 1** -- A custom evaluation rubric for agentic consulting AI
2. **Exercise 2** -- An LLM judge that uses your rubric to score responses automatically
3. **Exercise 3** -- A classification evaluation pipeline with sklearn metrics
4. **Optional 1** -- An A/B test runner for comparing system prompts
5. **Optional 2** -- A G-Eval scorer for consulting actionability
6. **Optional 3** -- An end-to-end evaluation pipeline combining all techniques

These tools form the foundation of a production evaluation system.
In later sessions you will integrate them with LangChain agents and RAG pipelines.